In [69]:
import numpy as np
import matplotlib as plt
import pandas as pd

In [70]:
df_owid = pd.read_csv("./tmpdatasets/owid-covid-data.csv")

In [71]:
df_who = pd.read_csv("./tmpdatasets/WHO-COVID-19-global-daily-data.csv")

In [72]:
# Step 1: Rename WHO columns
df_who = df_who.rename(columns={
    'Date_reported': 'date',
    'Country': 'location',
    'Country_code': 'iso_code',
    'New_cases': 'new_cases',
    'Cumulative_cases': 'total_cases',
    'New_deaths': 'new_deaths',
    'Cumulative_deaths': 'total_deaths'
})

# Step 2: Now select
df_who = df_who[['iso_code', 'location', 'date', 'new_cases', 'total_cases', 'new_deaths', 'total_deaths']]


In [73]:
owid_sel = df_owid[['iso_code', 'location', 'date', 'new_cases', 'total_cases', 'new_deaths', 'total_deaths']].copy()
who_sel = df_who[['iso_code', 'location', 'date', 'new_cases', 'total_cases', 'new_deaths', 'total_deaths']].copy()

In [74]:
# Mapping WHO location names to OWID equivalents
location_name_map = {
    'Bolivia (Plurinational State of)': 'Bolivia',
    'Bonaire, Sint Eustatius and Saba': 'Bonaire Sint Eustatius and Saba',
    'Brunei Darussalam': 'Brunei',
    'Cabo Verde': 'Cape Verde',
    'Côte d’Ivoire': "Cote d'Ivoire",
    'Democratic Republic of the Congo': 'Democratic Republic of Congo',
    'Democratic People\'s Republic of Korea': 'North Korea',
    'Falkland Islands (Malvinas)': 'Falkland Islands',
    'Holy See': 'Vatican',
    'Iran (Islamic Republic of)': 'Iran',
    'Kosovo (in accordance with UN Security Council Resolution 1244 (1999))': 'Kosovo',
    "Lao People's Democratic Republic": 'Laos',
    'Micronesia (Federated States of)': 'Micronesia (country)',
    'Netherlands (Kingdom of the)': 'Netherlands',
    'Republic of Korea': 'South Korea',
    'Republic of Moldova': 'Moldova',
    'Russian Federation': 'Russia',
    'R�union': 'Reunion',
    'Syrian Arab Republic': 'Syria',
    'Timor-Leste': 'East Timor',
    'Türkiye': 'Turkey',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'United Republic of Tanzania': 'Tanzania',
    'United States of America': 'United States',
    'Venezuela (Bolivarian Republic of)': 'Venezuela',
    'Viet Nam': 'Vietnam',
    'occupied Palestinian territory, including east Jerusalem': 'Palestine',
}


In [75]:
who_sel['location'] = who_sel['location'].replace(location_name_map)

In [76]:
owid_sel = owid_sel[~owid_sel['iso_code'].str.startswith('OWID_')]


In [77]:
# Unique country/location names
owid_countries = set(owid_sel['location'].unique())
who_countries = set(who_sel['location'].unique())

# Show mismatches (countries in one but not the other)
only_in_owid = owid_countries - who_countries
only_in_who = who_countries - owid_countries

print("🔴 Countries in OWID but not in WHO:", sorted(only_in_owid))
print("🔵 Countries in WHO but not in OWID:", sorted(only_in_who))

# Check ISO code mismatches
owid_iso = set(owid_sel['iso_code'].dropna().unique())
who_iso = set(who_sel['iso_code'].dropna().unique())

only_in_owid_iso = owid_iso - who_iso
only_in_who_iso = who_iso - owid_iso

print("🔴 ISO codes in OWID but not in WHO:", sorted(only_in_owid_iso))
print("🔵 ISO codes in WHO but not in OWID:", sorted(only_in_who_iso))


🔴 Countries in OWID but not in WHO: ['Curacao', 'Hong Kong', 'Macao', 'Taiwan', 'Western Sahara']
🔵 Countries in WHO but not in OWID: ['Cura�ao', 'International commercial vessel', 'International conveyance (American Samoa)', 'International conveyance (Diamond Princess)', 'International conveyance (Kiribati)', 'International conveyance (Solomon Islands)', 'International conveyance (Vanuatu)', 'Kosovo']
🔴 ISO codes in OWID but not in WHO: ['ABW', 'AFG', 'AGO', 'AIA', 'ALB', 'AND', 'ARE', 'ARG', 'ARM', 'ASM', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BES', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH', 'BLM', 'BLR', 'BLZ', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF', 'CAN', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COG', 'COK', 'COL', 'COM', 'CPV', 'CRI', 'CUB', 'CUW', 'CYM', 'CYP', 'CZE', 'DEU', 'DJI', 'DMA', 'DNK', 'DOM', 'DZA', 'ECU', 'EGY', 'ERI', 'ESH', 'ESP', 'EST', 'ETH', 'FIN', 'FJI', 'FLK', 'FRA', 'FRO', 'FSM', 'GAB', 'GBR', 'GEO', 'GGY', 'GHA', 'GIB', 'GIN', 'GLP

In [78]:
location_name_map.update({
    'Cura�ao': 'Curacao',
    'Hong Kong': 'Hong Kong',  # OWID has it, WHO likely missing or different name
    'Macao': 'Macao',
    'Taiwan': 'Taiwan',
    'Western Sahara': 'Western Sahara',
    'Kosovo': 'Kosovo',
    # International conveyances/vessels you might want to drop from WHO data
    'International commercial vessel': None,
    'International conveyance (American Samoa)': None,
    'International conveyance (Diamond Princess)': None,
    'International conveyance (Kiribati)': None,
    'International conveyance (Solomon Islands)': None,
    'International conveyance (Vanuatu)': None,
})
who_sel['location'] = who_sel['location'].replace(location_name_map)
who_sel = who_sel[who_sel['location'].notna()]  # Remove mapped to None rows

In [79]:
import pycountry
import pandas as pd

def iso3_to_iso2(iso3):
    try:
        country = pycountry.countries.get(alpha_3=iso3)
        if country:
            return country.alpha_2
        else:
            return None
    except Exception:
        return None

# Replace the 'iso_code' column in owid_sel with ISO2 codes converted from ISO3
owid_sel['iso_code'] = owid_sel['iso_code'].apply(lambda x: iso3_to_iso2(x) if pd.notna(x) else None)

# Optional: Check the result
print(owid_sel[['location', 'iso_code']].head())


      location iso_code
0  Afghanistan       AF
1  Afghanistan       AF
2  Afghanistan       AF
3  Afghanistan       AF
4  Afghanistan       AF


In [80]:
# Unique country/location names
owid_countries = set(owid_sel['location'].unique())
who_countries = set(who_sel['location'].unique())

# Show mismatches (countries in one but not the other)
only_in_owid = owid_countries - who_countries
only_in_who = who_countries - owid_countries

print("🔴 Countries in OWID but not in WHO:", sorted(only_in_owid))
print("🔵 Countries in WHO but not in OWID:", sorted(only_in_who))

# Check ISO code mismatches
owid_iso = set(owid_sel['iso_code'].dropna().unique())
who_iso = set(who_sel['iso_code'].dropna().unique())

only_in_owid_iso = owid_iso - who_iso
only_in_who_iso = who_iso - owid_iso

print("🔴 ISO codes in OWID but not in WHO:", sorted(only_in_owid_iso))
print("🔵 ISO codes in WHO but not in OWID:", sorted(only_in_who_iso))


🔴 Countries in OWID but not in WHO: ['Hong Kong', 'Macao', 'Taiwan', 'Western Sahara']
🔵 Countries in WHO but not in OWID: ['Kosovo']
🔴 ISO codes in OWID but not in WHO: ['EH', 'HK', 'MO', 'NA', 'TW']
🔵 ISO codes in WHO but not in OWID: ['XK']


In [81]:
who_sel.head()

,iso_code,location,date,new_cases,total_cases,new_deaths,total_deaths
0,VC,Saint Vincent and the Grenadines,2020-01-04,NaN,0,NaN,0
1,SN,Senegal,2020-01-04,NaN,0,NaN,0
2,SB,Solomon Islands,2020-01-04,0.0,0,0.0,0
3,LK,Sri Lanka,2020-01-04,0.0,0,0.0,0
4,SY,Syria,2020-01-04,NaN,0,NaN,0


In [82]:
owid_sel.head()

,iso_code,location,date,new_cases,total_cases,new_deaths,total_deaths
0,AF,Afghanistan,2020-01-05,0.0,0.0,0.0,0.0
1,AF,Afghanistan,2020-01-06,0.0,0.0,0.0,0.0
2,AF,Afghanistan,2020-01-07,0.0,0.0,0.0,0.0
3,AF,Afghanistan,2020-01-08,0.0,0.0,0.0,0.0
4,AF,Afghanistan,2020-01-09,0.0,0.0,0.0,0.0


In [83]:
# Remove rows where the location is in the mismatch lists
owid_sel_cleaned = owid_sel[~owid_sel['location'].isin(only_in_owid)]
who_sel_cleaned = who_sel[~who_sel['location'].isin(only_in_who)]

# Remove rows where the iso_code is in the mismatch lists
owid_sel_cleaned = owid_sel_cleaned[~owid_sel_cleaned['iso_code'].isin(only_in_owid_iso)]
who_sel_cleaned = who_sel_cleaned[~who_sel_cleaned['iso_code'].isin(only_in_who_iso)]

# Print the cleaned dataframes (or use them for further processing)
print("Cleaned OWID Data:")
print(owid_sel_cleaned.head())
print("Cleaned WHO Data:")
print(who_sel_cleaned.head())

# Drop 'Namibia' from WHO dataset
who_sel_cleaned = who_sel_cleaned[who_sel_cleaned['location'] != 'Namibia']





Cleaned OWID Data:
  iso_code     location        date  new_cases  total_cases  new_deaths  \
0       AF  Afghanistan  2020-01-05        0.0          0.0         0.0   
1       AF  Afghanistan  2020-01-06        0.0          0.0         0.0   
2       AF  Afghanistan  2020-01-07        0.0          0.0         0.0   
3       AF  Afghanistan  2020-01-08        0.0          0.0         0.0   
4       AF  Afghanistan  2020-01-09        0.0          0.0         0.0   

   total_deaths  
0           0.0  
1           0.0  
2           0.0  
3           0.0  
4           0.0  
Cleaned WHO Data:
  iso_code                          location        date  new_cases  \
0       VC  Saint Vincent and the Grenadines  2020-01-04        NaN   
1       SN                           Senegal  2020-01-04        NaN   
2       SB                   Solomon Islands  2020-01-04        0.0   
3       LK                         Sri Lanka  2020-01-04        0.0   
4       SY                             Syria  202

In [84]:
# Unique country/location names
owid_countries = set(owid_sel_cleaned['location'].unique())
who_countries = set(who_sel_cleaned['location'].unique())

# Show mismatches (countries in one but not the other)
only_in_owid = owid_countries - who_countries
only_in_who = who_countries - owid_countries

print("🔴 Countries in OWID but not in WHO:", sorted(only_in_owid))
print("🔵 Countries in WHO but not in OWID:", sorted(only_in_who))

# Check ISO code mismatches
owid_iso = set(owid_sel_cleaned['iso_code'].dropna().unique())
who_iso = set(who_sel_cleaned['iso_code'].dropna().unique())

only_in_owid_iso = owid_iso - who_iso
only_in_who_iso = who_iso - owid_iso

print("🔴 ISO codes in OWID but not in WHO:", sorted(only_in_owid_iso))
print("🔵 ISO codes in WHO but not in OWID:", sorted(only_in_who_iso))


🔴 Countries in OWID but not in WHO: []
🔵 Countries in WHO but not in OWID: []
🔴 ISO codes in OWID but not in WHO: []
🔵 ISO codes in WHO but not in OWID: []


In [85]:
# Save OWID cleaned dataset
owid_sel.to_csv("cleaned_owid_data.csv", index=False)

# Save WHO cleaned dataset
who_sel.to_csv("cleaned_who_data.csv", index=False)

